# Pola Rekursi Tingkat Lanjut di UnifyWeaver

Buku catatan ini mendemonstrasikan empat pola rekursi utama yang dapat dideteksi dan dioptimalkan oleh UnifyWeaver:

1. **Rekursi Ekor (Tail Recursion)** - Perulangan iteratif dengan akumulator
2. **Rekursi Linier (Linear Recursion)** - Panggilan rekursif tunggal dengan memoisasi
3. **Rekursi Pohon (Tree Recursion)** - Beberapa panggilan rekursif pada bagian-bagian struktur
4. **Rekursi Timbal Balik (Mutual Recursion)** - Predikat yang saling memanggil dalam siklus

## Tujuan Pembelajaran

- Memahami berbagai pola rekursi
- Melihat bagaimana UnifyWeaver mendeteksi dan mengoptimalkan setiap pola
- Membandingkan karakteristik kinerja
- Mempelajari kapan harus menggunakan setiap pola

## Pengaturan

Inisialisasi lingkungan UnifyWeaver.

In [ ]:
% Load initialization
['../init'].

% Load necessary modules
use_module(unifyweaver(core/recursive_compiler)).
use_module(unifyweaver(core/advanced/pattern_matchers)).

## Pola 1: Rekursi Ekor (Tail Recursion)

Rekursi ekor menggunakan akumulator untuk meneruskan hasil perantara, dan panggilan rekursif adalah **tindakan terakhir** dalam fungsi.

### Contoh: Menghitung Item dalam Daftar

In [ ]:
% Define tail-recursive count_items
:- dynamic count_items/3.

% Base case: empty list, return accumulator
count_items([], Acc, Acc).

% Recursive case: increment accumulator, recurse on tail
count_items([_|T], Acc, N) :-
    Acc1 is Acc + 1,
    count_items(T, Acc1, N).  % ← Tail position!

### Uji di Prolog

In [ ]:
% Test: count items in [a,b,c,d,e]
\+ \+ (
    count_items([a,b,c,d,e], 0, _N),
    format('Count: ~w~n', [_N])
).

### Periksa Deteksi Pola

In [ ]:
% Check if detected as tail recursive
\+ \+ (
    is_tail_recursive_accumulator(count_items/3, _AccInfo),
    format('Tail recursive: ~w~n', [_AccInfo])
).

### Kompilasi ke Bash

In [ ]:
% Compile and save
\+ \+ (
    compile_recursive(count_items/3, [], _BashCode),
    setup_call_cleanup(
        open('../output/count_items_demo.sh', write, _Stream),
        write(_Stream, _BashCode),
        close(_Stream)),
    writeln('✓ Compiled count_items to Bash with tail recursion optimization')
).

### Uji Bash yang Dihasilkan

In [ ]:
%%bash
source ../output/count_items_demo.sh
echo "Counting items in [a,b,c,d,e]:"
count_items "[a,b,c,d,e]" 0 ""

## Pola 2: Rekursi Linier (Linear Recursion)

Rekursi linier memiliki **tepat satu** panggilan rekursif per klausa, dengan komputasi terjadi setelah panggilan rekursif kembali.

### Contoh: Faktorial (Factorial)

In [ ]:
% Define factorial
:- dynamic factorial/2.

% Base case
factorial(0, 1).

% Recursive case: exactly ONE recursive call
factorial(N, F) :-
    N > 0,
    N1 is N - 1,
    factorial(N1, F1),  % ← One recursive call
    F is N * F1.        % ← Computation after call

### Uji di Prolog

In [ ]:
% Test: factorial of 5
\+ \+ (
    factorial(5, _F),
    format('5! = ~w~n', [_F])
).

### Periksa Deteksi Pola

In [ ]:
% Check if detected as linear recursive
is_linear_recursive_streamable(factorial/2),
writeln('✓ Detected as linear recursion').

### Kompilasi ke Bash

In [ ]:
% Compile and save
\+ \+ (
    compile_recursive(factorial/2, [], _BashCode),
    % Keep function definitions only; Brush treats sourced scripts as direct execution
    split_string(_BashCode, "\n", "\r", _BashLines),
    append(_LibraryLines, ["# Auto-execute when run directly (not when sourced)"|_], _BashLines),
    atomics_to_string(_LibraryLines, "\n", _LibraryCode),
    setup_call_cleanup(
        open('../output/factorial_demo.sh', write, _Stream),
        write(_Stream, _LibraryCode),
        close(_Stream)),
    writeln('✓ Compiled factorial to Bash with fold-based linear recursion')
).

### Uji Bash yang Dihasilkan

In [ ]:
%%bash
source ../output/factorial_demo.sh
echo "Factorial of 5:"
factorial 5 ""
echo ""
echo "Factorial of 10:"
factorial 10 ""

## Pola 3: Rekursi Pohon (Tree Recursion)

Rekursi pohon membuat **beberapa** panggilan rekursif untuk memproses berbagai bagian struktur.

### Contoh: Penjumlahan Pohon (Tree Sum)

In [ ]:
% Define tree_sum for binary trees
% Tree format: [Value, LeftSubtree, RightSubtree] or []
:- dynamic tree_sum/2.

% Base case: empty tree has sum 0
tree_sum([], 0).

% Recursive case: sum = value + left_sum + right_sum
tree_sum([V, L, R], Sum) :-
    tree_sum(L, LS),   % ← First recursive call
    tree_sum(R, RS),   % ← Second recursive call
    Sum is V + LS + RS.

### Uji di Prolog

In [ ]:
% Test: tree_sum of [5, [3, [1, [], []], []], [2, [], []]]
%       5
%      / \
%     3   2
%    /
%   1
\+ \+ (
    tree_sum([5, [3, [1, [], []], []], [2, [], []]], _Sum),
    format('Tree sum: ~w (expected 11)~n', [_Sum])
).

### Kompilasi ke Bash

In [ ]:
% Compile and save
\+ \+ (
    compile_recursive(tree_sum/2, [], _BashCode),
    setup_call_cleanup(
        open('../output/tree_sum_demo.sh', write, _Stream),
        write(_Stream, _BashCode),
        close(_Stream)),
    writeln('✓ Compiled tree_sum to Bash with tree recursion')
).

### Uji Bash yang Dihasilkan

In [ ]:
%%bash
source ../output/tree_sum_demo.sh
echo "Tree sum of [5,[3,[1,[],[]],[]],[2,[],[]]]:"
tree_sum "[5,[3,[1,[],[]],[]],[2,[],[]]]"

## Pola 4: Rekursi Timbal Balik (Mutual Recursion)

Rekursi timbal balik terjadi ketika dua atau lebih predikat saling memanggil dalam sebuah siklus.

### Contoh: Genap (Even) dan Ganjil (Odd)

In [ ]:
% Define mutually recursive is_even and is_odd
:- dynamic is_even/1.
:- dynamic is_odd/1.

% is_even base case
is_even(0).

% is_even recursive: N is even if N-1 is odd
is_even(N) :-
    N > 0,
    N1 is N - 1,
    is_odd(N1).  % ← Calls is_odd

% is_odd base case
is_odd(1).

% is_odd recursive: N is odd if N-1 is even
is_odd(N) :-
    N > 1,
    N1 is N - 1,
    is_even(N1).  % ← Calls is_even

### Uji di Prolog

In [ ]:
% Test even/odd
is_even(0), writeln('✓ 0 is even').
is_even(4), writeln('✓ 4 is even').
is_odd(3), writeln('✓ 3 is odd').
is_odd(7), writeln('✓ 7 is odd').

### Periksa Rekursi Timbal Balik

In [ ]:
% Build call graph and find SCCs
\+ \+ (
    use_module(unifyweaver(core/advanced/call_graph)),
    use_module(unifyweaver(core/advanced/scc_detection)),

    build_call_graph([is_even/1, is_odd/1], _Graph),
    format('Call graph: ~w~n', [_Graph]),

    find_sccs(_Graph, _SCCs),
    format('SCCs (mutual recursion groups): ~w~n', [_SCCs])
).

### Kompilasi ke Bash

In [ ]:
% Compile the mutual recursion group
\+ \+ (
    use_module(unifyweaver(core/advanced/mutual_recursion)),

    compile_mutual_recursion([is_even/1, is_odd/1], [], _BashCode),
    split_string(_BashCode, "\n", "\r", _BashLines),
    append(_LibraryLines, ["# Main dispatch: route command line calls to functions"|_], _BashLines),
    atomics_to_string(_LibraryLines, "\n", _LibraryCode),
    setup_call_cleanup(
        open('../output/even_odd_demo.sh', write, _Stream),
        write(_Stream, _LibraryCode),
        close(_Stream)),
    writeln('✓ Compiled is_even/is_odd to Bash with shared memoization')
).

### Uji Bash yang Dihasilkan

In [ ]:
%%bash
source ../output/even_odd_demo.sh
echo "Testing is_even and is_odd:"
is_even 0 >/dev/null && echo "✓ 0 is even"
is_even 4 >/dev/null && echo "✓ 4 is even"
is_odd 3 >/dev/null && echo "✓ 3 is odd"
is_odd 7 >/dev/null && echo "✓ 7 is odd"
is_even 5 >/dev/null 2>&1 || echo "✓ 5 is not even"

## Perbandingan Pola

Mari kita bandingkan karakteristik setiap pola:

| Pola | Panggilan Rekursif | Optimasi | Kompleksitas Ruang | Paling Cocok Untuk |
|:--------|:----------------|:-------------|:-----------------|:---------|
| **Ekor** | 1 (pada posisi ekor) | Perulangan iteratif | O(1) | Akumulator, pemindaian linier |
| **Linier** | 1 (posisi mana saja) | Fold + memoisasi | Tabel memo O(n) | Fibonacci, faktorial |
| **Pohon** | 2+ (bagian struktur) | Dekomposisi struktural | Tumpukan O(kedalaman) | Operasi pohon/graf |
| **Timbal Balik** | 1+ (lintas predikat) | Memoisasi bersama | Tabel bersama O(n) | Genap/ganjil, definisi bersama |

## Urutan Deteksi Pola

UnifyWeaver mencoba mencocokkan pola dalam urutan ini:

1. **Rekursi Ekor** (paling efisien)
2. **Rekursi Linier** (kecuali dilarang)
3. **Rekursi Pohon** (struktural)
4. **Rekursi Timbal Balik** (deteksi SCC)
5. **Rekursi Dasar** (cadangan default)

Anda dapat memengaruhi deteksi dengan `forbid_linear_recursion/1`.

## Latihan: Giliran Anda!

Cobalah mendefinisikan dan mengompilasi predikat-predikat berikut:

### 1. Penjumlahan Rekursi Ekor
```prolog
sum_list([], Acc, Acc).
sum_list([H|T], Acc, Sum) :-
    Acc1 is Acc + H,
    sum_list(T, Acc1, Sum).
```

### 2. Fibonacci Rekursi Linier
```prolog
fib(0, 0).
fib(1, 1).
fib(N, F) :-
    N > 1,
    N1 is N - 1,
    N2 is N - 2,
    fib(N1, F1),
    fib(N2, F2),
    F is F1 + F2.
```

### 3. Tinggi Pohon
```prolog
tree_height([], 0).
tree_height([_, L, R], H) :-
    tree_height(L, HL),
    tree_height(R, HR),
    H is max(HL, HR) + 1.
```

In [ ]:
% Your code here!


## Ringkasan

Dalam buku catatan ini, Anda telah mempelajari:

✅ Empat pola rekursi utama di UnifyWeaver

✅ Cara mendefinisikan setiap pola di Prolog

✅ Cara UnifyWeaver mendeteksi dan mengoptimalkan setiap pola

✅ Karakteristik kinerja setiap pola

✅ Kapan harus menggunakan setiap pola

## Langkah Selanjutnya

Lanjutkan ke **Buku Catatan 3: Visualisasi Graf Panggilan** untuk mempelajari analisis dan visualisasi kode tingkat lanjut!